In [ ]:
# 1 ----------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

folder = Path("/Users/stephenpeschel/Desktop/Hydro/camels_ch/timeseries/observation_based/")
gauge_id = 2263  # Chamuerabach (La Punt-Chamues-ch)
camels_file = folder / f"CAMELS_CH_obs_based_{gauge_id}.csv"
assert camels_file.exists(), f"File not found: {camels_file}"

latitude = 46.6
col_P, col_T = "precipitation(mm/d)", "temperature_mean(degC)"
col_Tmin, col_Tmax = "temperature_min(degC)", "temperature_max(degC)"
col_Q = "discharge_spec(mm/d)"

# Start with the best parameters from Exercise 3 (we calibrate them in step 2)
params = dict(k=0.05, S_max=400.0, T_snow=1.0, DDF=5.0, I_thr=1.0)

# The three periods of the exercise
warmup = slice("1981-01-01", "1982-12-31")
calib = slice("1983-01-01", "2009-12-31")
valid = slice("2010-01-01", "2020-12-31")

In [ ]:
# 2 ----------
## Data and PET
df = pd.read_csv(camels_file, parse_dates=["date"], index_col="date")
df = df.loc["1981-01-01":"2020-12-31"]

for c in [col_T, col_Tmin, col_Tmax]:  # fill short temperature gaps
    df[c] = df[c].interpolate(limit=5)

def hargreaves_pet(tmin, tmean, tmax, doy, lat):
    phi = np.radians(lat)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))
    ra = (24 * 60 / np.pi) * 0.0820 * dr * (
            ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws))
    pet = 0.0023 * 0.408 * ra * (tmean + 17.8) * np.sqrt(np.clip(tmax - tmin, 0, None))
    return np.clip(pet, 0, None)

df["PET"] = hargreaves_pet(df[col_Tmin], df[col_T], df[col_Tmax],
                           df.index.day_of_year.values, latitude)

P = df[col_P].fillna(0).values
T = df[col_T].values
PET = df["PET"].values
assert not np.isnan(T).any() and not np.isnan(PET).any(), "T or PET still has NaN (long gap?)"

In [ ]:
# 3 ----------
## Model (same as Exercise 3)
def run_model(P, T, PET, k=0.1, S_max=150.0, T_snow=1.0, DDF=3.0, I_thr=1.0):
    SWE, S = 0.0, 0.5 * S_max  # snowpack, flowerpot
    Q, SWE_list = [], []
    for t in range(len(P)):
        # Rain or snow
        if T[t] < T_snow:
            snow, rain = P[t], 0.0
        else:
            snow, rain = 0.0, P[t]
        # Snow: accumulation and melt
        SWE = SWE + snow
        melt = min(DDF * max(T[t], 0), SWE)
        SWE = SWE - melt
        # Interception: first I_thr mm of rain is intercepted and evaporates
        intercepted = min(rain, I_thr)
        through = rain - intercepted
        ET_int = min(intercepted, PET[t])
        PET_left = PET[t] - ET_int
        if SWE > 0:
            PET_left = 0.0  # no soil evaporation under snow
        # Flowerpot
        S = S + through + melt
        overflow = max(S - S_max, 0)
        S = S - overflow
        ET = min(PET_left * (S / S_max) ** 0.5, S)
        S = S - ET
        q = k * S
        S = S - q
        Q.append(q + overflow)
        SWE_list.append(SWE)
    return np.array(Q), np.array(SWE_list)

In [ ]:
# 4 ----------
## Objective functions
def evaluate(obs, sim):
    ok = obs.notna()  # only days where Q was measured
    o, s = obs[ok].values, sim[ok].values
    # NSE: 1 = perfect, 0 = no better than the mean. Dominated by the high flows.
    nse = 1 - np.sum((o - s) ** 2) / np.sum((o - o.mean()) ** 2)
    # Nash-log: NSE on log(Q), so the low flows count as much as the peaks
    lo, ls = np.log(o + 0.01), np.log(s + 0.01)  # + 0.01 so we never take log(0)
    nse_log = 1 - np.sum((lo - ls) ** 2) / np.sum((lo - lo.mean()) ** 2)
    # Bias: too much (+) or too little (-) water in total, in %
    bias = 100 * (s.sum() - o.sum()) / o.sum()
    return nse, nse_log, bias

In [ ]:
# 5 ----------
## Run the whole period (warm-up included) and score each period separately
q, swe = run_model(P, T, PET, **params)
df["Q_sim"] = q
df["SWE"] = swe

print("Parameters:", params)
print(f"{'period':12s} {'NSE':>6s} {'Nash-log':>9s} {'Bias %':>7s}")
for name, period in [("calibration", calib), ("validation", valid)]:
    nse, nse_log, bias = evaluate(df.loc[period, col_Q], df.loc[period, "Q_sim"])
    print(f"{name:12s} {nse:6.2f} {nse_log:9.2f} {bias:7.1f}")

In [ ]:
# 6 ----------
## Hydrograph of the whole period, with the three periods shaded
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df[col_Q], "k", lw=0.6, label="observed")
ax.plot(df.index, df["Q_sim"], "r", lw=0.6, label="modelled")
ax.axvspan(pd.Timestamp(warmup.start), pd.Timestamp(warmup.stop), color="gray", alpha=0.3, label="warm-up")
ax.axvspan(pd.Timestamp(calib.start), pd.Timestamp(calib.stop), color="tab:green", alpha=0.1, label="calibration")
ax.axvspan(pd.Timestamp(valid.start), pd.Timestamp(valid.stop), color="tab:blue", alpha=0.1, label="validation")
ax.set_ylabel("Q (mm/d)")
ax.legend(loc="upper left", ncol=5, fontsize=8)
plt.tight_layout();
plt.savefig("fig_ex4_periods.png", dpi=150);
plt.show()

day = df["Q_sim"].idxmax()
print(day.date(), "Q_sim", round(df.loc[day, "Q_sim"], 1), "T", round(df.loc[day, col_T], 1),
      "SWE before", round(df["SWE"].shift(1)[day]))

In [ ]:
# 7 ----------
## Step 2: automatic calibration (grid search) on the calibration period only
# We try every combination of the values below, run the model from 1981
# (so the warm-up is included), and score only 1983-2009.
# The score is the average of NSE (high flows) and Nash-log (low flows).
# Change the weights in "score = ..." if you want to favor one of them.

import itertools, time

grid = dict(k=[0.01, 0.02, 0.05, 0.1],
            S_max=[50, 100, 200, 400],
            DDF=[2, 3, 4, 5],
            T_snow=[0.0, 1.0, 2.0],
            I_thr=[0.5, 1.0, 2.0])

obs_cal = df.loc[calib, col_Q]
in_cal = (df.index >= calib.start) & (df.index <= calib.stop)

results = []
t0 = time.time()
for values in itertools.product(*grid.values()):
    p = dict(zip(grid.keys(), values))
    q, _ = run_model(P, T, PET, **p)
    nse, nse_log, bias = evaluate(obs_cal, pd.Series(q[in_cal], index=obs_cal.index))
    score = 0.5 * nse + 0.5 * nse_log
    results.append(dict(p, NSE=nse, NashLog=nse_log, Bias=bias, score=score))
print(f"Tried {len(results)} parameter sets in {time.time() - t0:.0f} s")

results = pd.DataFrame(results).sort_values("score", ascending=False)
print("\nTop 10 parameter sets (calibration 1983-2009):")
print(results.head(10).round(2).to_string(index=False))

best = results.iloc[0]
params = {name: float(best[name]) for name in grid}
print("\nBest parameters:", params)

In [ ]:
# 8 ----------
## run again with the best parameters and evaluate calibration AND validation
q, swe = run_model(P, T, PET, **params)
df["Q_sim"] = q
df["SWE"] = swe

print(f"\n{'period':12s} {'NSE':>6s} {'Nash-log':>9s} {'Bias %':>7s}")
for name, period in [("calibration", calib), ("validation", valid)]:
    nse, nse_log, bias = evaluate(df.loc[period, col_Q], df.loc[period, "Q_sim"])
    print(f"{name:12s} {nse:6.2f} {nse_log:9.2f} {bias:7.1f}")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df[col_Q], "k", lw=0.6, label="observed")
ax.plot(df.index, df["Q_sim"], "r", lw=0.6, label="modelled (calibrated)")
ax.axvspan(pd.Timestamp(warmup.start), pd.Timestamp(warmup.stop), color="gray", alpha=0.3, label="warm-up")
ax.axvspan(pd.Timestamp(calib.start), pd.Timestamp(calib.stop), color="tab:green", alpha=0.1, label="calibration")
ax.axvspan(pd.Timestamp(valid.start), pd.Timestamp(valid.stop), color="tab:blue", alpha=0.1, label="validation")
ax.set_ylabel("Q (mm/d)")
ax.legend(loc="upper left", ncol=5, fontsize=8)
plt.tight_layout();
plt.savefig("fig_ex4_calibrated.png", dpi=150);
plt.show()

In [ ]:
# 9 ----------
## Step 3: quick runoff (overland flow)
# Problem without it: once the flowerpot is full (S_max), all extra water
# overflows and leaves the catchment on the SAME day -> unrealistic spikes.
# Now the overflow goes into a small "quick" reservoir F that drains a share
# k_fast of its water each day (e.g. 0.5 = half per day). So a big overflow
# becomes a peak spread over a few days, like real overland / fast flow.
def run_model_qr(P, T, PET, k=0.1, S_max=150.0, T_snow=1.0, DDF=3.0, I_thr=1.0, k_fast=0.5):
    SWE, S, F = 0.0, 0.5 * S_max, 0.0  # snowpack, flowerpot, quick reservoir
    Q, SWE_list = [], []
    for t in range(len(P)):
        # Rain or snow
        if T[t] < T_snow:
            snow, rain = P[t], 0.0
        else:
            snow, rain = 0.0, P[t]
        # Snow: accumulation and melt
        SWE = SWE + snow
        melt = min(DDF * max(T[t], 0), SWE)
        SWE = SWE - melt
        # Interception
        intercepted = min(rain, I_thr)
        through = rain - intercepted
        ET_int = min(intercepted, PET[t])
        PET_left = PET[t] - ET_int
        if SWE > 0:
            PET_left = 0.0
        # Flowerpot (slow flow)
        S = S + through + melt
        overflow = max(S - S_max, 0)
        S = S - overflow
        ET = min(PET_left * (S / S_max) ** 0.5, S)
        S = S - ET
        q_slow = k * S
        S = S - q_slow
        # Quick reservoir (fast flow): receives the overflow
        F = F + overflow
        q_fast = k_fast * F
        F = F - q_fast
        Q.append(q_slow + q_fast)
        SWE_list.append(SWE)
    return np.array(Q), np.array(SWE_list)

In [ ]:
# 10 ----------
## Calibrate again, now with k_fast as an extra parameter
# k, S_max and T_snow now go further, because step 2 picked values at the edge
# of its grid (k = 0.01, S_max = 400, T_snow = 0). Takes about 30 s.
grid_qr = dict(k=[0.005, 0.01, 0.02, 0.05],
               S_max=[100, 200, 400, 800],
               DDF=[2, 3, 4],
               T_snow=[-1.0, 0.0, 1.0],
               I_thr=[1.0, 2.0],
               k_fast=[0.1, 0.2, 0.4, 0.7])

results_qr = []
t0 = time.time()
for values in itertools.product(*grid_qr.values()):
    p = dict(zip(grid_qr.keys(), values))
    q, _ = run_model_qr(P, T, PET, **p)
    nse, nse_log, bias = evaluate(obs_cal, pd.Series(q[in_cal], index=obs_cal.index))
    # average of NSE and Nash-log, minus a penalty for the volume error
    # (a bias of -20 % costs 0.2), so the model can't just lose water
    score = 0.5 * nse + 0.5 * nse_log - abs(bias) / 100
    results_qr.append(dict(p, NSE=nse, NashLog=nse_log, Bias=bias, score=score))
print(f"\nQuick runoff model: tried {len(results_qr)} parameter sets in {time.time() - t0:.0f} s")

results_qr = pd.DataFrame(results_qr).sort_values("score", ascending=False)
print("\nTop 10 parameter sets WITH quick runoff (calibration 1983-2009):")
print(results_qr.head(10).round(3).to_string(index=False))

best_qr = results_qr.iloc[0]
params_qr = {name: float(best_qr[name]) for name in grid_qr}
print("\nBest parameters with quick runoff:", params_qr)

In [ ]:
# 11 ----------
## Compare: without vs with quick runoff
q_qr, _ = run_model_qr(P, T, PET, **params_qr)
df["Q_sim_qr"] = q_qr

print(f"\n{'model':22s} {'period':12s} {'NSE':>6s} {'Nash-log':>9s} {'Bias %':>7s}")
for model_name, col in [("without quick runoff", "Q_sim"), ("with quick runoff", "Q_sim_qr")]:
    for name, period in [("calibration", calib), ("validation", valid)]:
        nse, nse_log, bias = evaluate(df.loc[period, col_Q], df.loc[period, col])
        print(f"{model_name:22s} {name:12s} {nse:6.2f} {nse_log:9.2f} {bias:7.1f}")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df[col_Q], "k", lw=0.6, label="observed")
ax.plot(df.index, df["Q_sim_qr"], "r", lw=0.6, label="modelled (with quick runoff)")
ax.axvspan(pd.Timestamp(warmup.start), pd.Timestamp(warmup.stop), color="gray", alpha=0.3, label="warm-up")
ax.axvspan(pd.Timestamp(calib.start), pd.Timestamp(calib.stop), color="tab:green", alpha=0.1, label="calibration")
ax.axvspan(pd.Timestamp(valid.start), pd.Timestamp(valid.stop), color="tab:blue", alpha=0.1, label="validation")
ax.set_ylabel("Q (mm/d)")
ax.legend(loc="upper left", ncol=5, fontsize=8)
plt.tight_layout();
plt.savefig("fig_ex4_quickrunoff.png", dpi=150);
plt.show()

In [ ]:
# 12 ----------
## Step 4: snow correction factor (SCF)
# Rain gauges catch too little snow (wind blows it past the gauge), so the
# measured snowfall is too small. SCF multiplies the snowfall:
# SCF = 1.2 means "20 % more snow than measured". Rain is not changed.
# This adds the water the model was missing (negative Bias).
def run_model_scf(P, T, PET, k=0.1, S_max=150.0, T_snow=1.0, DDF=3.0, I_thr=1.0,
                  k_fast=0.5, SCF=1.0):
    SWE, S, F = 0.0, 0.5 * S_max, 0.0  # snowpack, flowerpot, quick reservoir
    Q, SWE_list = [], []
    for t in range(len(P)):
        # Rain or snow (only the snow is corrected)
        if T[t] < T_snow:
            snow, rain = P[t] * SCF, 0.0
        else:
            snow, rain = 0.0, P[t]
        # Snow: accumulation and melt
        SWE = SWE + snow
        melt = min(DDF * max(T[t], 0), SWE)
        SWE = SWE - melt
        # Interception
        intercepted = min(rain, I_thr)
        through = rain - intercepted
        ET_int = min(intercepted, PET[t])
        PET_left = PET[t] - ET_int
        if SWE > 0:
            PET_left = 0.0
        # Flowerpot (slow flow)
        S = S + through + melt
        overflow = max(S - S_max, 0)
        S = S - overflow
        ET = min(PET_left * (S / S_max) ** 0.5, S)
        S = S - ET
        q_slow = k * S
        S = S - q_slow
        # Quick reservoir (fast flow)
        F = F + overflow
        q_fast = k_fast * F
        F = F - q_fast
        Q.append(q_slow + q_fast)
        SWE_list.append(SWE)
    return np.array(Q), np.array(SWE_list)


In [ ]:
# 13 ----------
## Calibrate again with SCF
# Last time S_max = 800, T_snow = -1 and k_fast = 0.1 were at the edge of the
# grid, so the ranges go further now. I_thr hardly changed the score, so it is
# fixed at 1.0 to save time. Takes about 30-40 s.
# 2nd run: k, DDF and k_fast were at the edge, so their ranges go further.
# S_max, T_snow and SCF were in the middle, so they keep fewer values around
# the best one. Takes about 45-60 s.
grid_scf = dict(k=[0.002, 0.003, 0.005, 0.01],
                S_max=[200, 400, 800],
                DDF=[4, 5, 6, 7],
                T_snow=[-2.0, -1.0, 0.0],
                I_thr=[1.0],
                k_fast=[0.02, 0.03, 0.05, 0.1],
                SCF=[1.2, 1.4, 1.6])

results_scf = []
t0 = time.time()
for values in itertools.product(*grid_scf.values()):
    p = dict(zip(grid_scf.keys(), values))
    q, _ = run_model_scf(P, T, PET, **p)
    nse, nse_log, bias = evaluate(obs_cal, pd.Series(q[in_cal], index=obs_cal.index))
    score = 0.5 * nse + 0.5 * nse_log - abs(bias) / 100
    results_scf.append(dict(p, NSE=nse, NashLog=nse_log, Bias=bias, score=score))
print(f"\nSCF model: tried {len(results_scf)} parameter sets in {time.time() - t0:.0f} s")

results_scf = pd.DataFrame(results_scf).sort_values("score", ascending=False)
print("\nTop 10 parameter sets WITH quick runoff + SCF (calibration 1983-2009):")
print(results_scf.head(10).round(3).to_string(index=False))

best_scf = results_scf.iloc[0]
params_scf = {name: float(best_scf[name]) for name in grid_scf}
print("\nBest parameters with SCF:", params_scf)

# check: is a best value at the edge of its range? Then the range should go further.
for name, values in grid_scf.items():
    if len(values) > 1 and params_scf[name] in (min(values), max(values)):
        print(f"  note: {name} = {params_scf[name]} is at the edge of its range {values}")

In [ ]:
# 14 ----------
## Compare all three versions
q_scf, swe_scf = run_model_scf(P, T, PET, **params_scf)
df["Q_sim_scf"] = q_scf

print(f"\n{'model':22s} {'period':12s} {'NSE':>6s} {'Nash-log':>9s} {'Bias %':>7s}")
for model_name, col in [("without quick runoff", "Q_sim"),
                        ("with quick runoff", "Q_sim_qr"),
                        ("quick runoff + SCF", "Q_sim_scf")]:
    for name, period in [("calibration", calib), ("validation", valid)]:
        nse, nse_log, bias = evaluate(df.loc[period, col_Q], df.loc[period, col])
        print(f"{model_name:22s} {name:12s} {nse:6.2f} {nse_log:9.2f} {bias:7.1f}")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df[col_Q], "k", lw=0.6, label="observed")
ax.plot(df.index, df["Q_sim_scf"], "r", lw=0.6, label="modelled (quick runoff + SCF)")
ax.axvspan(pd.Timestamp(warmup.start), pd.Timestamp(warmup.stop), color="gray", alpha=0.3, label="warm-up")
ax.axvspan(pd.Timestamp(calib.start), pd.Timestamp(calib.stop), color="tab:green", alpha=0.1, label="calibration")
ax.axvspan(pd.Timestamp(valid.start), pd.Timestamp(valid.stop), color="tab:blue", alpha=0.1, label="validation")
ax.set_ylabel("Q (mm/d)")
ax.legend(loc="upper left", ncol=5, fontsize=8)
plt.tight_layout();
plt.savefig("fig_ex4_scf.png", dpi=150);
plt.show()

# Zoom on 3 years, so you can see single peaks and recessions
zoom = slice("2012-01-01", "2014-12-31")
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.loc[zoom].index, df.loc[zoom, col_Q], "k", lw=0.8, label="observed")
ax.plot(df.loc[zoom].index, df.loc[zoom, "Q_sim_scf"], "r", lw=0.8, label="modelled (quick runoff + SCF)")
ax.set_ylabel("Q (mm/d)");
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout();
plt.savefig("fig_ex4_scf_zoom.png", dpi=150);
plt.show()